In [1]:
import gurobipy as gp
from gurobipy import GRB, nlfunc
from utils.data import *

In [3]:
start= dt.datetime(year=2000, month=1, day=1)
end= dt.datetime(year=2023, month=1, day=1)
all_stocks = ['AAPL', "GOOG", "NVDA", "CL", "BA", "NKE", "DB"]

n = len(all_stocks)

mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)
#Sigma_XY = Sigma[]

r = 0.05
T = 90
S = [yf.Ticker(s).history(period="1d") for s in all_stocks]

In [69]:
m = gp.Model()

S = [yf.Ticker(s).history(period="1d").Close.to_list()[0] for s in all_stocks]
sigma = [Sigma[i, i] for i in range(n)]

w = m.addMVar(n, lb=0, name="weight")

mu_d = m.addMVar(n, lb=-GRB.INFINITY, name="expectation")
Delta = m.addMVar(n, lb=0.3, ub=.7, name="Delta")
Phi_inv = m.addMVar(n, lb=-GRB.INFINITY, name="phi of inverse CDF")
phi = m.addMVar(n, name="phi")
K = m.addMVar(n, lb=0, name="strike")
P = m.addMVar(n, lb=-GRB.INFINITY, name="put price")
d1 = m.addMVar(n, lb=0, ub=1, name="d1")
d2 = m.addMVar(n, lb=0, ub=1, name="d2")

t = m.addVar()
var_norm = m.addVar()

m.addConstr(gp.quicksum([wi for wi in w]) == 1)
m.addConstr(var_norm**2 == w.T @ Sigma @ w)
m.addConstr(gp.quicksum([mu_d[i]*w[i] for i in range(n)]) >= var_norm * t)
m.addConstrs(mu_d[i] == mu[i] + sigma[i]*(Delta[i]*Phi_inv[i] + phi[i]) - P[i]/S[i]\
                            for i in range(n))

#m.addConstrs(Phi_inv[i] == -1/1.7*nlfunc.log(1/Delta[i]-1) for i in range(n))
m.addConstrs(Phi_inv[i] == 4/1.7*(Delta[i]-1/2) for i in range(n))

m.addConstrs(phi[i] == 1/np.sqrt(2*np.pi)*(1-Phi_inv[i]**2/2) for i in range(n))
m.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))

#m.addConstrs(d1[i] == (1-K[i]/S[i]+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))
#m.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))

#m.addConstrs(P[i] >= K[i]*np.exp(-r*T)*(1/2-1/np.sqrt(2*np.pi)*d2[i]) - S[i]*(1/2-1/np.sqrt(2*np.pi)*d1[i]) for i in range(n))

m.setObjective(t, GRB.MAXIMIZE)

m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Fedora Linux 42 (Workstation Edition)")

CPU model: AMD Ryzen 9 5900X 12-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 15 rows, 65 columns and 35 nonzeros
Model fingerprint: 0x72818091
Model has 16 quadratic constraints
Coefficient statistics:
  Matrix range     [8e-03, 2e+00]
  QMatrix range    [3e-04, 1e+00]
  QLMatrix range   [3e-03, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [3e-01, 1e+00]
  RHS range        [1e+00, 3e+02]
  QRHS range       [3e-03, 4e-01]
Presolve removed 7 rows and 21 columns

Continuous model is non-convex -- solving as a MIP

Presolve removed 14 rows and 49 columns
Presolve time: 0.00s
Presolved: 127 rows, 54 columns, 283 nonzeros
Presolved model has 37 bilinear constraint(s)
         in product terms.
         Presolve was not able to compute smaller bounds for these variables.
   

In [66]:
weights = np.zeros(n)

var = m.getVars()

weights = [v.X for v in var if "weight" in v.VarName]

deltas = [v.X for v in var if "Delta" in v.VarName]
P = [v.X for v in var if "price" in v.VarName]
d = [v.X for v in var if "d1" in v.VarName]
weights, deltas, P, d

([0.5,
  0.002052987244322013,
  0.0001701977230195491,
  0.4523822230430816,
  0.045379355625782436,
  6.6612672852909896e-06,
  8.575096509089164e-06],
 [0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3],
 [22.706831539768707,
  -322774188648816.0,
  26.40829266843039,
  1.336133355400483,
  8.609865835231052,
  2.0249004481975232,
  0.19274450321897763],
 [10.459933992754534,
  17.80833978418138,
  4.6093001430771245,
  59.82436175772942,
  15.258141450696982,
  27.6586891981687,
  11.966722707117961])

In [50]:
import plotly.express as px

px.line(benchmark(all_stocks, weights, "SPY", end, dt.datetime.today()))

In [56]:
px.line(benchmark(all_stocks, np.ones(len(all_stocks))/len(all_stocks), "SPY", end, dt.datetime.today()))

In [31]:
sigma = [np.std(get_hist(s, start, end).Open.to_numpy()) for s in all_stocks]

sigma

[np.float64(41.13958930297493),
 np.float64(33.92487904583442),
 np.float64(5.754463949391831),
 np.float64(19.614596993522884),
 np.float64(94.9451142725771),
 np.float64(37.677191413275544),
 np.float64(18.383465051812426)]